# VPT demo — predict flower images with the trained model (no training)
**Course:** CS4046E Deep Learning for Computer Vision · NIT Calicut · **Paper:** *Visual Prompt Tuning*, Jia et al., ECCV 2022 · **Code:** the authors' official repository, embedded unmodified.

### Just import this one file and press *Run All*
Needs: **Internet On** (Kaggle: Session options) for the one-time download of the official ViT-B/16 backbone weights (about 413 MB). A GPU is not needed.
Everything else is **inside this notebook** (demo code, official repository, the trained prompts + heads of VPT-deep, VPT-shallow, Linear and Bias).

* Without any image of your own, the notebook shows the model on random images of the Oxford Flowers-102 **test** split (data it never trained on).
* **To classify the examiner's images:** on Kaggle use *Add Input → Upload*, select the image files and create the dataset; then run section 4 again (the images are found automatically). On a PC put them in a folder `sir_images/` next to the notebook.

**Limits to keep in mind:** the model knows only the **102 flower categories**; for any other picture it still names the closest flower, and a high probability does **not** prove the picture is a flower.

## 1. Embedded project (unmodified official code + trained parameters)

In [ ]:
# collapsed: base64 of the project zip (demo code, official repository, trained checkpoints)
PROJECT_ZIP_SHA256 = "cc2ee73b6c5b38a8fbb73f3dafd413670fef96bd5c108be64494b6092c0f4589"
PROJECT_ZIP_B64 = """
...
8GwtAAAA
"""

## 2. Setup

In [ ]:
import base64, hashlib, importlib.util, io, os, subprocess, sys, zipfile
from pathlib import Path

# 2a) install the few libraries the official code needs (no-op when they are already present)
for module, pip_name in [("torch", "torch"), ("torchvision", "torchvision"), ("timm", "timm"), ("fvcore", "fvcore"), ("iopath", "iopath"),
                         ("yacs", "yacs"), ("ml_collections", "ml_collections"), ("simplejson", "simplejson"),
                         ("termcolor", "termcolor"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("PIL", "pillow")]:
    if importlib.util.find_spec(module) is None:
        subprocess.call([sys.executable, "-m", "pip", "install", "-q", pip_name])

# 2b) use the project folder if the notebook sits inside it (PC), otherwise unpack the embedded copy
if Path("demo.py").exists() and Path("third_party").exists():
    root = Path.cwd()
else:
    blob = base64.b64decode("".join(PROJECT_ZIP_B64.split()))
    assert hashlib.sha256(blob).hexdigest() == PROJECT_ZIP_SHA256, "embedded project is corrupted"
    base = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extractall(base)
    root = base / "vpt-flowers-demo"
os.chdir(root); sys.path.insert(0, str(root))

# 2c) prove the official code is byte-identical to the original (hash list shipped with the project)
bad = [rel for line in Path("MANIFEST.sha256").read_text().splitlines()
       for digest, rel in [line.split("  ", 1)] if hashlib.sha256(Path(rel).read_bytes()).hexdigest() != digest]
print("project folder:", root)
print("official code verified:", "all files byte-identical to the original" if not bad else f"MODIFIED: {bad}")

In [ ]:
import demo
demo.download_weights()        # skipped when weights/imagenet21k_ViT-B_16.npz already exists (about 413 MB, one time)

## 3. Load the trained model (VPT-deep — the best prompt method of this project)

In [ ]:
METHOD = "vpt_deep"            # also embedded: vpt_shallow, linear, bias
predictor = demo.Predictor(METHOD)
print(predictor.describe())

## 4. Predict the examiner's images
Add the images (Kaggle: *Add Input → Upload*; PC: folder `sir_images/`), then run this cell again. Unreadable files are skipped with a warning.

In [ ]:
IMAGE_DIR = None               # optional: a specific folder with images; otherwise ./sir_images or an attached Kaggle dataset is used

paths = demo.collect_images([], IMAGE_DIR) if IMAGE_DIR else demo.find_user_images()

if paths:
    results = predictor.predict(paths, topk=5)
    demo.print_predictions(predictor, results)
    fig = demo.plot_results(results, title=f"{predictor.entry['label']} - top-5 predictions")
else:
    print("No images supplied yet -> section 6 shows the model on random test images instead.\n"
          "To classify your own: attach them (Kaggle: Add Input -> Upload) or put them in ./sir_images, then re-run this cell.")

## 5. (Optional) Compare the embedded trained methods on the same images

In [ ]:
if paths:
    for m in ["vpt_shallow", "linear", "bias"]:
        p = demo.Predictor(m)
        r = p.predict(paths, topk=1)
        print(f"{p.entry['label']:<12s} (test acc {p.entry['test_acc']:.2f}%): " +
              " | ".join(f"{Path(x['path']).name}: {x['top'][0][0]} {100 * x['top'][0][1]:.0f}%" for x in r))
        del p
else:
    print("(skipped - no images supplied)")

## 6. The model on random images of the Oxford Flowers-102 test split (data it never trained on)
Runs automatically when no images were supplied (downloads the ~330 MB test split once). Set `FORCE = True` to run it anyway.

In [ ]:
FORCE = False
if FORCE or not paths:
    try:
        import numpy as np, torchvision
        from flower_names import FLOWER_NAMES
        ds = torchvision.datasets.Flowers102("data", split="test", download=True)
        pick = np.random.default_rng(0).choice(len(ds), 8, replace=False)
        out = predictor.predict([Path(ds._image_files[i]) for i in pick], topk=3)
        hits = 0
        for r, i in zip(out, pick):
            truth, pred = FLOWER_NAMES[int(ds._labels[i])], r["top"][0][0]; hits += truth == pred
            print(f"{'OK ' if truth == pred else 'ERR'} true: {truth:<26s} predicted: {pred:<26s} {100 * r['top'][0][1]:.0f}%")
        print(f"{hits}/8 correct on this random sample (full test accuracy: {predictor.entry['test_acc']:.2f}% on 6,149 images)")
        fig = demo.plot_results(out, title="random Oxford Flowers-102 test images")
    except Exception as e:                          # no internet / dataset host unreachable: keep the notebook usable
        print(f"Could not load the test split ({type(e).__name__}: {e}). Use section 4 with your own images instead.")